# 03 — Custom Objects from URDF

In this notebook you will:
1. Learn what a URDF file is
2. Write a simple URDF for a coloured box
3. Load it into the simulation with `add_object_from_urdf()`
4. Understand supported shapes and limitations

---

## What is a URDF?

URDF (*Unified Robot Description Format*) is an XML file that describes a robot or object:
its geometry, mass, visual appearance, and collision boundaries.

Here is the minimal structure for a **single rigid object**:

```xml
<?xml version="1.0"?>
<robot name="my_object">
  <link name="base_link">

    <!-- Visual: what the object looks like -->
    <visual>
      <geometry>
        <box size="0.05 0.05 0.05"/>   <!-- width depth height (metres) -->
      </geometry>
      <material name="red">
        <color rgba="0.8 0.2 0.2 1.0"/>
      </material>
    </visual>

    <!-- Collision: the physics boundary (can be simpler than visual) -->
    <collision>
      <geometry>
        <box size="0.05 0.05 0.05"/>
      </geometry>
    </collision>

  </link>
</robot>
```

Supported collision shapes: `box`, `sphere`, `cylinder`, `mesh`.

## Step 1 — Write a URDF file

The cell below writes a URDF for a **red box** to disk.

In [ ]:
box_urdf = """\
<?xml version="1.0"?>
<robot name="red_box">
  <link name="base_link">

    <visual>
      <geometry>
        <box size="0.06 0.06 0.06"/>
      </geometry>
      <material name="red">
        <color rgba="0.85 0.15 0.15 1.0"/>
      </material>
    </visual>

    <collision>
      <geometry>
        <box size="0.06 0.06 0.06"/>
      </geometry>
    </collision>

    <inertial>
      <mass value="0.1"/>
      <inertia ixx="0.0001" ixy="0" ixz="0"
               iyy="0.0001" iyz="0"
               izz="0.0001"/>
    </inertial>

  </link>
</robot>
"""

urdf_path = "/tmp/red_box.urdf"
with open(urdf_path, "w") as f:
    f.write(box_urdf)

print(f"URDF written to: {urdf_path}")

## Step 2 — Load the URDF into the simulation

In [ ]:
from sawyer_student import SawyerSim

sim = SawyerSim(gui=True)

# Load the URDF — the centre of the box will be at z=0.03 (half of 6 cm)
sim.add_object_from_urdf(
    urdf_path=urdf_path,
    position=[0.45, 0.0, 0.03],
    name="red_box",
)

sim.start()

You should see a red box sitting on the floor in the viewer.
Let's do a simple pick to confirm it behaves correctly.

In [ ]:
sim.open_gripper()
sim.move_to(0.45, 0.0, 0.25)   # approach
sim.move_to(0.45, 0.0, 0.04, speed="slow")  # descend
sim.close_gripper()
sim.wait(0.8)
sim.move_to(0.45, 0.0, 0.30)   # lift
sim.wait(1.0)

In [ ]:
sim.open_gripper()
sim.wait(0.5)
sim.reset()
sim.close()

---

## Sphere URDF example

In [ ]:
sphere_urdf = """\
<?xml version="1.0"?>
<robot name="blue_sphere">
  <link name="base_link">
    <visual>
      <geometry><sphere radius="0.04"/></geometry>
      <material name="blue"><color rgba="0.1 0.3 0.9 1.0"/></material>
    </visual>
    <collision>
      <geometry><sphere radius="0.04"/></geometry>
    </collision>
  </link>
</robot>
"""

with open("/tmp/blue_sphere.urdf", "w") as f:
    f.write(sphere_urdf)

print("Sphere URDF written.")

## Cylinder URDF example

In [ ]:
cylinder_urdf = """\
<?xml version="1.0"?>
<robot name="green_cylinder">
  <link name="base_link">
    <visual>
      <geometry>
        <cylinder radius="0.03" length="0.10"/>
      </geometry>
      <material name="green"><color rgba="0.2 0.75 0.2 1.0"/></material>
    </visual>
    <collision>
      <geometry>
        <cylinder radius="0.03" length="0.10"/>
      </geometry>
    </collision>
  </link>
</robot>
"""

with open("/tmp/green_cylinder.urdf", "w") as f:
    f.write(cylinder_urdf)

print("Cylinder URDF written.")

## Using all three together

In [ ]:
sim2 = SawyerSim(gui=True)

sim2.add_object_from_urdf("/tmp/red_box.urdf",       [0.45,  0.15, 0.03], name="box")
sim2.add_object_from_urdf("/tmp/blue_sphere.urdf",   [0.45,  0.00, 0.04], name="sphere")
sim2.add_object_from_urdf("/tmp/green_cylinder.urdf",[0.45, -0.15, 0.05], name="cylinder")

sim2.start()
sim2.wait(3.0)  # admire the scene
sim2.close()

---

## How to create URDF from Blender or CAD

### From Blender
1. Model your object in Blender.
2. Export as **Collada (`.dae`)** or **STL (`.stl`)**.
3. Write a URDF that references the mesh:

```xml
<geometry>
  <mesh filename="my_object.dae" scale="1 1 1"/>
</geometry>
```

4. Put the URDF and the mesh file in the same folder.
5. Call `sim.add_object_from_urdf("path/to/my_object.urdf", ...)`.

### From FreeCAD / SolidWorks
Both tools have URDF export plugins.  Search for:
- FreeCAD: `freecad-ros2` or `assembly4 URDF`
- SolidWorks: `sw2urdf` exporter

---

## Supported shapes and limitations

| URDF shape | Supported |
|---|---|
| `<box>` | ✅ |
| `<sphere>` | ✅ |
| `<cylinder>` | ✅ |
| `<mesh filename=...>` | ✅ (STL, DAE, OBJ) |
| Multi-link URDF | ⚠️ Only first link with collision is used |
| Joints between links | ❌ Not supported (use single-link URDF) |

**Tip:** Keep your object URDF to a single `<link>` for best results.